In [ ]:
from constants import *
from astropy import units as u

from phoenix_grid_creator.spectral_grid import spectral_grid
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))
# fits_file_paths =fits_file_paths[:1]
resolution : u.Quantity[u.K] = .001*u.um
spec_grid : spectral_grid = spectral_grid.from_local_raw(fits_file_paths, resolution, False) # need to downsample resolution otherwise the spectral grid won't fit in memory...

# units are getting lost somewhere, idk why
spec_grid.Wavelengths = spec_grid.Wavelengths.to(u.um)
spec_grid.Fluxes *= u.Jy

In [ ]:
spec_grid.get_spectrum(spec_grid.T_effs[-1], spec_grid.FeHs[0], spec_grid.Log_gs[0]).plot()

In [ ]:


%matplotlib inline
# take 2 random spectra
# combine them with random weights
# see if you can reach a global minimum at a given resolution
# repeat for many resolutions

number_of_components : int = 3

from matplotlib import pyplot as plt

import rich
from spectrum_component_analyser.mcmc.simulated_spectra import get_random_simulated_spectrum
from spectrum_component_analyser.spectral_component import spectral_component
from spectrum_component_analyser.spectrum import spectrum

true_components, component_spectra, combined_spectrum = get_random_simulated_spectrum(number_of_components, spec_grid)

In [ ]:


true_table = spectral_component.return_default_table()
for c in true_components:
    c.pretty_print(true_table)
rich.print(true_table)

In [ ]:
"""
https://www.aanda.org/articles/aa/pdf/2016/03/aa22261-13.pdf talks about a dynamical mask, could maybe introduce that?

e.g. using

total_fluxes : np.ndarray = np.zeros((len(spec_grid.Wavelengths))) * u.Jy

all_phoenix_params = list(itertools.product(spec_grid.T_effs, spec_grid.FeHs, spec_grid.Log_gs))

for t, f, l in all_phoenix_params:
    total_fluxes += spec_grid.LookupTable[t, f, l]

average_flux = total_fluxes / len(all_phoenix_params)

average_flux = average_flux[mask]
"""

import numpy as np
import scipy as sp
from spectrum_component_analyser.interpolated_spectrum import get_interpolated_phoenix_spectrum

number_of_parameters : int = 4

def chi_squared_error(params):
    observed = combined_spectrum.Fluxes.copy()
    simulated : np.ndarray = []
    
    for i in range(number_of_components):
        weight = params[0 + i * number_of_parameters]
        t = params[1 + i * number_of_parameters]
        f = params[2 + i * number_of_parameters]
        l = params[3 + i * number_of_parameters]

        t *= u.K
        f *= u.dex
        l *= u.dex

        spec : phoenix_spectrum = get_interpolated_phoenix_spectrum(t, f, l, star_name="simulated spectrum", spec_grid=spec_grid)

        if len(simulated) == 0:
            simulated = weight * spec.Fluxes
        else:
            simulated += weight * spec.Fluxes

    e = np.sum(((observed - simulated)**2).value)

    return e

parameter_bounds = [
        (.0, 2.),
        (np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)),
        (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)),
        # (target.feh.value * .9, target.feh.value * 1.1),
        # (0, 0),
        (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value)),
        # (target.log_g.value * .9, target.log_g.value * 1.1)
        # (1.9, 2.1),
]

initial_guess = [
    1,
    3000,
    0.0,
    0.0
]

r : sp.optimize.OptimizeResult = sp.optimize.differential_evolution(
    func=chi_squared_error,
    # x0 = initial_guess * number_of_components,
    bounds=parameter_bounds * number_of_components,
    callback=print,
    maxiter=100
)

print(r)

In [ ]:
%matplotlib inline

print("found")

total_flux = []

for i in range(number_of_components):
    w_title : str = f"weight {i}"
    t_title : str = f"teff {i}"
    f_title : str = f"feh {i}"
    l_title : str = f"log_g {i}"

    w = r.x[0 + number_of_parameters*i]
    t = r.x[1 + number_of_parameters*i] * u.K
    f = r.x[2  + number_of_parameters*i] * u.dex
    l = r.x[3 + number_of_parameters*i] * u.dex

    print(f"{w_title:<20} : {w}")
    print(f"{t_title:<20} : {t}")
    print(f"{f_title:<20} : {f}")
    print(f"{l_title:<20} : {l}")
    print("\n")

    
    componentised_spectrum = get_interpolated_phoenix_spectrum(t, f, l, star_name= f"simulated spectrum component {i}", spec_grid=spec_grid)

    componentised_spectrum.Fluxes *= w

    componentised_spectrum.plot(clear=False)

    if len(total_flux) == 0:
        total_flux = componentised_spectrum.Fluxes
    else:
        total_flux += componentised_spectrum.Fluxes

print("simulated / true")
rich.print(true_table)

plt.legend()
plt.show()

# plot components neatly
# make subplot with: literature belief parameters & residuals, our 1 component solution & residuals, our 2 / 3 etc component solution

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(30, 16), sharex=True)

my_result : spectrum = spectrum(
    wavelengths = combined_spectrum.Wavelengths,
    fluxes=total_flux,
    normalised_point=None,
    observational_resolution=None,
    observational_wavelengths=None,
    temperature=None,
    name="my result",
    normalise=False
)

# normalise everything for now as im unsure
combined_spectrum.normalise_flux()
my_result.normalise_flux()

ax1.plot(combined_spectrum.Wavelengths, combined_spectrum.Fluxes, label="simulated spectrum", color="black")
ax1.plot(my_result.Wavelengths, my_result.Fluxes, label="my result", color="orange")

my_residuals = np.abs(my_result.Fluxes - combined_spectrum.Fluxes) / combined_spectrum.Fluxes

# ax2.plot(combined_spectrum.Wavelengths, literature_residuals, label="literature", color="orange")
ax2.plot(combined_spectrum.Wavelengths, my_residuals, label="my result", color="blue")
ax2.set_ylabel(r"Residual = $\frac{\mathrm{Fitted\ Flux}-\mathrm{Observed\ Flux}}{\mathrm{Observed\ Flux}}$")

ax1.legend()
ax2.legend()

plt.show()

print(np.sum(np.abs(my_residuals)))

In [ ]:
import numpy as np
import emcee
import corner # Great for visualizing the results
from spectrum_component_analyser.interpolated_spectrum import get_interpolated_phoenix_spectrum
from spectrum_component_analyser.mcmc.helper import MCMCHelper

# --- Config ---


mcmc : MCMCHelper = MCMCHelper(
    parameter_bounds=parameter_bounds,
    number_of_components=number_of_components,
    number_of_parameters=number_of_parameters,
    observed_spectrum=combined_spectrum,
    spec_grid=spec_grid,
    nsteps = 5000,
    nwalkers=64
)

# --- Execution ---

sampler, samples = mcmc.run(r)


In [ ]:
%matplotlib inline

mcmc.plot(sampler, samples, true_components)

In [ ]:
import numpy as np
import rich
from astropy.units import Quantity

# 1. Calculate the 16th, 50th, and 84th percentiles for all parameters
# axis=0 calculates these across the flattened samples
percentiles = np.percentile(samples, [16, 50, 84], axis=0)

# 2. Prepare the table structure
results_table = spectral_component.return_default_table()

teffs = [percentiles[1, 1 + i * number_of_parameters] for i in range(number_of_components)]

# Get indices for descending order (highest weight first)
sorted_indices = np.argsort(teffs)[::-1]
print(teffs)
# 3. Process each component
for i in sorted_indices:
    index_offset = i * number_of_parameters
    
    # We create a little helper to get (median, lower_err, upper_err)
    def get_stats(idx):
        low, med, high = percentiles[:, idx]
        # Average the lower and upper bounds for a symmetric plus-minus
        # or use (high-med) if you prefer reporting the upper bound
        err = (high - low) / 2
        return med, err

    # Extract median and uncertainty
    weight_med, weight_err = get_stats(index_offset)
    teff_med, teff_err     = get_stats(index_offset + 1)
    feh_med, feh_err       = get_stats(index_offset + 2)
    logg_med, logg_err     = get_stats(index_offset + 3)

    # 4. Format strings with rounding to 2 decimal places and plus-minus
    # We store these in a way that respects your object's structure 
    # but provides the visual "plus-minus" clarity
    
    # Example format: "5780.12 ± 15.43"
    display_weight = f"{weight_med:.2f} ± {weight_err:.2f}"
    display_teff   = f"{teff_med:.2f} ± {teff_err:.2f} K"
    display_feh    = f"{feh_med:.2f} ± {feh_err:.2f} dex"
    display_logg   = f"{logg_med:.2f} ± {logg_err:.2f} dex"

    # NOTE: If your pretty_print() method strictly requires Floats/Quantities,
    # you may need to modify it to accept these strings or add a custom row manually.
    # Here, we instantiate the component with medians for logic, 
    # but we'll print the row with uncertainties.
    
    recovered_component : spectral_component = spectral_component(
        teff_med * u.K, 
        feh_med * u.dex, 
        logg_med * u.dex, 
        weight_med
    )

    # Use your class's existing method to add to the table
    recovered_component.pretty_print(results_table)
    
    # Hack to overwrite the last row's cells with our +/- strings 
    # (Assuming the table is a rich.table.Table)
    last_row_idx = len(results_table.rows) - 1
    results_table.columns[0]._cells[last_row_idx] = display_weight
    results_table.columns[1]._cells[last_row_idx] = display_teff
    results_table.columns[2]._cells[last_row_idx] = display_feh
    results_table.columns[3]._cells[last_row_idx] = display_logg

# 5. Display
print("\n[MCMC RECOVERED PARAMETERS WITH 1-SIGMA ERRORS]")
rich.print(results_table)

print("\n[ORIGINAL PARAMETERS]")
rich.print(true_table)

In [ ]:
import matplotlib.pyplot as plt
import astropy.units as u
import numpy as np

%matplotlib widget
# 1. Extract Median Parameters
best_params : np.ndarray = np.median(samples, axis=0)

# 2. Setup Plotting Environment
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(30, 16), sharex=True, 
                               gridspec_kw={'height_ratios': [3, 1]})
plt.subplots_adjust(hspace=0.05)

# Initialize total flux with correct units
total_flux : u.Quantity = np.zeros(len(combined_spectrum.Wavelengths)) * u.Jy

# 3. Component Generation
colors = ["#3498db", "#e74c3c", "#2ecc71"]

for i in range(number_of_components):
    idx = i * number_of_parameters
    
    # Extracting with explicit units from your style
    w = best_params[idx]
    t = best_params[idx + 1] * u.K
    f = best_params[idx + 2] * u.dex
    l = best_params[idx + 3] * u.dex
    
    mcmc_component = get_interpolated_phoenix_spectrum(
        t, f, l, 
        star_name=f"MCMC Component {i}", 
        spec_grid=spec_grid
    )

    # Scale flux by the weight
    weighted_flux = mcmc_component.Fluxes * w
    total_flux += weighted_flux

    # weighted_flux /= np.sum(weighted_flux.value)

    # Plot individual component
    ax1.plot(
        combined_spectrum.Wavelengths, 
        weighted_flux.to(u.Jy), 
        label=f"Component {i}: {t:.0f}, {f:.1f}FeH", 
        color=colors[i % len(colors)], 
        alpha=0.5, 
        linestyle='--'
    )

# 4. Create Result Spectrum Object (matching your style)
my_result : spectrum = spectrum(
    wavelengths=combined_spectrum.Wavelengths,
    fluxes=total_flux,
    normalised_point=None,
    name="MCMC Median Fit",
    normalise=False,
    observational_resolution=None,
    observational_wavelengths=None,
    temperature=None
)

# 5. Normalization & Comparison
# Assuming we normalise to the mean for a stable residual
obs_flux_norm = combined_spectrum.Fluxes
fit_flux_norm = my_result.Fluxes

ax1.plot(combined_spectrum.Wavelengths, obs_flux_norm, label="Fake Data", color="black", linewidth=1.5)
ax1.plot(my_result.Wavelengths, fit_flux_norm, label="MCMC Fit", color="orange", linewidth=2)

# 6. Residuals (Dimensionless)
my_residuals = (fit_flux_norm - obs_flux_norm) / obs_flux_norm

ax2.plot(combined_spectrum.Wavelengths, my_residuals.value, color="blue", alpha=0.7)
ax2.axhline(0, color='black', linestyle=':', alpha=0.5)

# Formatting labels with units
ax1.set_ylabel(f"Flux")
ax1.legend(loc="upper right", ncol=2)
ax2.set_ylabel(r"$\frac{F_{fit} - F_{obs}}{F_{obs}}$")
ax2.set_xlabel(f"Wavelength ({combined_spectrum.Wavelengths.unit})")

# Smart zoom for residuals
res_min, res_max = np.percentile(my_residuals.value, [1, 99])
ax2.set_ylim(res_min - 0.01, res_max + 0.01)

plt.show()

print(f"Total Absolute Residual: {np.sum(np.abs(my_residuals.value))}")